# Problem 1

# Problem 2

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# --------------------------
# 1. INPUT DATA

stations = [1, 2, 3, 4, 5, 6, 7, 8]

# OD Demand Matrix (8x8): demand_data[i-1][j-1] = # of passengers i->j
demand_data = [
    [0,   150, 300, 200, 100, 250, 180, 220],  # from station 1
    [160, 0,   210, 270, 130, 190, 240, 150],  # from station 2
    [240, 180, 0,   310, 140, 260, 170, 200],  # from station 3
    [200, 220, 190, 0,   280, 210, 320, 160],  # from station 4
    [130, 140, 150, 160, 0,   230, 250, 290],  # from station 5
    [210, 170, 260, 230, 310, 0,   180, 150],  # from station 6
    [190, 240, 200, 270, 220, 160, 0,   210],  # from station 7
    [250, 200, 150, 180, 210, 230, 190, 0  ]   # from station 8
]

# Travel-time matrix (minutes): time_data[i-1][j-1] = travel time i->j
time_data = [
    [0,  10, 15, 20, 25, 30, 35, 40],  # from station 1
    [12, 0,  18, 24, 16, 20, 28, 32],  # from station 2
    [14, 16, 0,  22, 26, 30, 18, 24],  # from station 3
    [20, 24, 26, 0,  12, 18, 22, 28],  # from station 4
    [18, 14, 20, 16, 0,  15, 25, 30],  # from station 5
    [25, 20, 22, 28, 30, 0,  14, 19],  # from station 6
    [30, 28, 24, 20, 18, 15, 0,  17],  # from station 7
    [35, 32, 28, 24, 20, 18, 16, 0 ]   # from station 8
]

# Convert matrices into dictionaries for easy lookup.
demand = {}
timeM = {}
for i in stations:
    for j in stations:
        demand[(i, j)] = demand_data[i-1][j-1]
        timeM[(i, j)] = time_data[i-1][j-1]

# Problem parameters
bus_capacity   = 150
max_buses      = 5
total_time_cap = 391890
fixed_cost     = 1000
station_cost   = 100

# --------------------------
# 2. ENUMERATE ROUTES (ASCENDING AND DESCENDING)
# Generate all no backtracking routes from i to j
all_routes = []
for i in stations:
    for j in stations:
        if i != j:
            # Determine the lower and higher station values.
            lo = min(i, j)
            hi = max(i, j)
            intermediate = []
            for m in stations:
                if m > lo and m < hi:
                    intermediate.append(m)
            
            # There are 2^(number of intermediates) subsets of intermediate stations.
            n_mid = len(intermediate)
            num_subsets = 1 << n_mid  # 2^n_mid
            
            subset_mask = 0
            while subset_mask < num_subsets:
                chosen = []
                chosen.append(i)  # starting station
                
                # Add selected intermediate stations.
                for idx in range(n_mid):
                    bit = 1 << idx
                    if (subset_mask & bit) != 0:
                        chosen.append(intermediate[idx])
                
                chosen.append(j)  # ending station
                
                # Sort chosen stations to be monotonic.
                if i < j:
                    chosen.sort()            # ascending order
                else:
                    chosen.sort(reverse=True)  # descending order
                
                # Save the route as a tuple.
                all_routes.append(tuple(chosen))
                subset_mask += 1

# Remove duplicates and sort the list for a stable order.
all_routes = list(set(all_routes))
all_routes.sort()
R = range(len(all_routes))  # Route indices

# --------------------------
# 3. PRECOMPUTE COST & TRAVEL TIMES
# --------------------------
route_cost = {}
route_length = {}
# Tdict[((i, j), r)] holds the travel time in minutes for OD pair (i, j) on route r.
Tdict = {}

for r_idx in R:
    route_stations = all_routes[r_idx]
    length_r = len(route_stations)
    route_cost[r_idx] = fixed_cost + station_cost * length_r
    route_length[r_idx] = length_r
    
    cumulative_time = {}
    first_s = route_stations[0]
    cumulative_time[first_s] = 0
    total_so_far = 0
    
    for idx in range(len(route_stations) - 1):
        s1 = route_stations[idx]
        s2 = route_stations[idx + 1]
        seg_time = timeM[(s1, s2)]
        total_so_far += seg_time
        cumulative_time[s2] = total_so_far
    
    for (i_val, j_val) in demand.keys():
        if i_val != j_val:
            if i_val in route_stations and j_val in route_stations:
                idx_i = route_stations.index(i_val)
                idx_j = route_stations.index(j_val)
                if idx_i < idx_j:
                    Tdict[((i_val, j_val), r_idx)] = cumulative_time[j_val] - cumulative_time[i_val]
                else:
                    Tdict[((i_val, j_val), r_idx)] = 0
            else:
                Tdict[((i_val, j_val), r_idx)] = 0
        else:
            Tdict[((i_val, j_val), r_idx)] = 0

# Build a list of all distinct OD pairs (i, j) with i != j.
K = []
for i in stations:
    for j in stations:
        if i != j:
            K.append((i, j))

# --------------------------
# 4. BUILD THE MODEL
# --------------------------
model = gp.Model("BiDirectional_Route_Model")

# Decision variables:
# y_vars[r]: binary variable indicating whether route r is used.
# b_vars[r]: integer number of buses assigned to route r (0 to max_buses).
# x_vars[(i,j), r]: continuous variable representing passenger flow for OD (i,j) on route r.
y_vars = {}
b_vars = {}
x_vars = {}

for r_idx in R:
    y_vars[r_idx] = model.addVar(vtype=GRB.BINARY, name="y_r%d" % r_idx)
    b_vars[r_idx] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=max_buses, name="b_r%d" % r_idx)

for (i_val, j_val) in K:
    for r_idx in R:
        x_vars[((i_val, j_val), r_idx)] = model.addVar(vtype=GRB.CONTINUOUS, lb=0,
                                                        name="x_%d_%d_r%d" % (i_val, j_val, r_idx))

# --------------------------
# 5. SET THE OBJECTIVE
# --------------------------
# Minimize the total cost of used routes.
obj_expr = gp.LinExpr()
for r_idx in R:
    obj_expr.addTerms(route_cost[r_idx], y_vars[r_idx])
model.setObjective(obj_expr, GRB.MINIMIZE)

# --------------------------
# 6. ADD CONSTRAINTS
# --------------------------
# (a) Demand satisfaction: For every OD pair, the sum over routes must equal the demand.
for (i_val, j_val) in K:
    lhs = gp.LinExpr()
    for r_idx in R:
        lhs.addTerms(1.0, x_vars[((i_val, j_val), r_idx)])
    model.addConstr(lhs == demand[(i_val, j_val)], "demand_%d_%d" % (i_val, j_val))

# (b) Capacity constraint: Passenger flow on route r cannot exceed bus capacity times number of buses.
for r_idx in R:
    lhs_cap = gp.LinExpr()
    for (i_val, j_val) in K:
        lhs_cap.addTerms(1.0, x_vars[((i_val, j_val), r_idx)])
    model.addConstr(lhs_cap <= bus_capacity * b_vars[r_idx], "cap_r%d" % r_idx)

# (c) Bus-route activation: Number of buses is zero if route is not used.
for r_idx in R:
    model.addConstr(b_vars[r_idx] <= max_buses * y_vars[r_idx], "buslink_r%d" % r_idx)

# (d) Total travel time constraint.
lhs_time = gp.LinExpr()
for (i_val, j_val) in K:
    for r_idx in R:
        val_time = Tdict[((i_val, j_val), r_idx)]
        if val_time > 0:
            lhs_time.addTerms(val_time, x_vars[((i_val, j_val), r_idx)])
model.addConstr(lhs_time <= total_time_cap, "time_limit")

# (e) Force x_vars to zero if the route does not serve an OD pair.
for (i_val, j_val) in K:
    for r_idx in R:
        if Tdict[((i_val, j_val), r_idx)] == 0:
            model.addConstr(x_vars[((i_val, j_val), r_idx)] == 0,
                            "no_service_%d_%d_r%d" % (i_val, j_val, r_idx))

# --------------------------
# 7. SOLVE THE MODEL
# --------------------------
model.optimize()

# --------------------------
# 8. PRINT THE RESULTS
# --------------------------
if model.status == GRB.OPTIMAL:
    print("\nOPTIMAL SOLUTION FOUND.")
    print("Objective (min cost) =", model.ObjVal)
    
    used_routes = []
    for r_idx in R:
        if y_vars[r_idx].X > 0.5:
            used_routes.append(r_idx)
    
    print("Number of routes used =", len(used_routes))
    for r_idx in used_routes:
        route_stations = all_routes[r_idx]
        print("  Route index =", r_idx, "stations =", route_stations)
        print("    y =", y_vars[r_idx].X)
        print("    buses =", b_vars[r_idx].X)
        print("    route cost =", route_cost[r_idx])
        
        # Print passenger assignments for this route.
        for (i_val, j_val) in K:
            val = x_vars[((i_val, j_val), r_idx)].X
            if val > 1e-6:
                print("OD (%d -> %d): %.1f passengers" % (i_val, j_val, val))
        print()
    
    # Calculate total passenger travel time used.
    total_time_used = 0.0
    for (i_val, j_val) in K:
        for r_idx in R:
            flow = x_vars[((i_val, j_val), r_idx)].X
            if flow > 1e-6:
                total_time_used += Tdict[((i_val, j_val), r_idx)] * flow
    print("Total passenger travel time =", total_time_used, "(limit =", total_time_cap, ")")
else:
    print("No optimal solution found. Solver status =", model.status)

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 25125 rows, 28652 columns and 84474 nonzeros
Model fingerprint: 0xf09c17fb
Variable types: 27664 continuous, 988 integer (494 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 2e+03]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 4e+05]
Presolve removed 24574 rows and 24574 columns
Presolve time: 0.02s
Presolved: 551 rows, 4078 columns, 9704 nonzeros
Variable types: 3584 continuous, 494 integer (494 binary)
Found heuristic solution: objective 71800.000000

Root relaxation: objective 2.085089e+04, 1195 iterations, 0.01 seconds (0.01 work units)

Interrupt request received

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0   